# 1. STANDARTIZATION

add all 7 dedup csv files from folder "raw"


*   normalizes column names
*   cleans text whitespaces
*   normalizes source names
*   choses appropriate text colums
*   creates unique article IDs
*   standartizes data types
*   text lenghts











In [ ]:
%%writefile step1_standardize_all.py
from __future__ import annotations

import re
import html
from pathlib import Path
import pandas as pd

INPUT_DIR = Path("/content/raw")
OUTPUT_DIR = Path("/content/standardized")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_COLUMNS = [
    "id",
    "title",
    "full_text",
    "source",
    "publication_date",
    "topic",
    "class_label",
    "url",
    "text_length",
]

BASE_REQUIRED_COLUMNS = [
    "source",
    "publication_date",
    "topic",
    "class_label",
    "url",
]


def normalize_column_name(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"\s+", "_", name)
    return name


def clean_basic_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = html.unescape(text)
    text = text.replace("\ufeff", "")
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_source(value: object) -> str:
    source = clean_basic_text(value).lower()
    source_map = {
        "delfi": "delfi",
        "tvnet": "tvnet",
        "lsm": "lsm",
        "ltv": "ltv",
        "rebaltica": "rebaltica",
        "re:baltica": "rebaltica",
        "recheck": "recheck",
        "re:check": "recheck",
        "atmaskots": "delfi_atmaskots",
        "delfi atmaskots": "delfi_atmaskots",
        "tvnet faktomāts": "tvnet_faktomats",
        "tvnet faktomats": "tvnet_faktomats",
        "rebaltica recheck": "rebaltica_recheck",
        "re:baltica re:check": "rebaltica_recheck",
    }
    return source_map.get(source, source)


def create_article_id(source: str, index: int) -> str:
    safe_source = re.sub(r"[^a-z0-9_]+", "_", str(source).lower()).strip("_")
    if not safe_source:
        safe_source = "unknown"
    return f"{safe_source}_{index:05d}"


def choose_text_columns(df: pd.DataFrame) -> tuple[str, str]:
    """
    Reliable class (0):
        title_clean + full_text_clean
    Misinformation class (1):
        title_masked + full_text_masked
    """
    if "class_label" not in df.columns:
        raise ValueError("Missing class_label column")

    class_values = set(pd.to_numeric(df["class_label"], errors="coerce").dropna().astype(int).unique())

    if class_values == {0}:
        title_col = "title_clean"
        full_text_col = "full_text_clean"
    elif class_values == {1}:
        title_col = "title_masked"
        full_text_col = "full_text_masked"
    else:
        raise ValueError(
            f"Unexpected class_label values: {sorted(class_values)}. "
            "Each input file should contain only one class."
        )

    missing = [c for c in [title_col, full_text_col] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for selected class: {missing}")

    return title_col, full_text_col


def standardize_file(file_path: Path):
    print(f"\nProcessing: {file_path.name}")
    df = pd.read_csv(file_path, encoding="utf-8")
    df.columns = [normalize_column_name(c) for c in df.columns]

    missing_base = [c for c in BASE_REQUIRED_COLUMNS if c not in df.columns]
    if missing_base:
        print(f"Skipping {file_path.name} (missing columns: {missing_base})")
        return

    try:
        selected_title_col, selected_text_col = choose_text_columns(df)
    except ValueError as e:
        print(f"Skipping {file_path.name} ({e})")
        return

    # Build unified model-facing text columns
    df["title"] = df[selected_title_col].apply(clean_basic_text)
    df["full_text"] = df[selected_text_col].apply(clean_basic_text)

    # Clean metadata
    df["topic"] = df["topic"].apply(clean_basic_text)
    df["url"] = df["url"].apply(clean_basic_text)
    df["source"] = df["source"].apply(normalize_source)
    df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
    df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype("Int64")

    # Final text length must reflect the selected full_text column
    df["text_length"] = df["full_text"].fillna("").astype(str).str.len()

    # Create stable IDs
    ids = []
    for i, row in df.iterrows():
        src = row["source"] if isinstance(row["source"], str) else "unknown"
        ids.append(create_article_id(src, i + 1))

    df.insert(0, "id", ids)

    # Keep only the unified final schema
    df = df[FINAL_COLUMNS].copy()

    output_file = OUTPUT_DIR / f"{file_path.stem}_standardized.csv"
    df.to_csv(output_file, index=False, encoding="utf-8")

    print(f"Saved: {output_file.name}")
    print(f"Rows: {len(df)}")
    print(f"Using columns: {selected_title_col} + {selected_text_col}")


def run_all():
    files = sorted(INPUT_DIR.glob("*.csv"))

    if not files:
        print("No CSV files found in /content/raw")
        return

    print(f"Found {len(files)} files")

    for f in files:
        standardize_file(f)

    print("\nAll files processed.")


if __name__ == "__main__":
    run_all()

Writing step1_standardize_all.py


In [ ]:
!python step1_standardize_all.py

Found 7 files

Processing: delfi_atmaskots_dedup.csv
Saved: delfi_atmaskots_dedup_standardized.csv
Rows: 430
Using columns: title_masked + full_text_masked

Processing: delfi_reliable_dedup.csv
Saved: delfi_reliable_dedup_standardized.csv
Rows: 577
Using columns: title_clean + full_text_clean

Processing: lsm_arzemes_dedup.csv
Saved: lsm_arzemes_dedup_standardized.csv
Rows: 348
Using columns: title_clean + full_text_clean

Processing: ltv_reliable_dedup.csv
Saved: ltv_reliable_dedup_standardized.csv
Rows: 30
Using columns: title_clean + full_text_clean

Processing: rebaltica_recheck_dedup.csv
Saved: rebaltica_recheck_dedup_standardized.csv
Rows: 441
Using columns: title_masked + full_text_masked

Processing: tvnet_faktomats_dedup.csv
Saved: tvnet_faktomats_dedup_standardized.csv
Rows: 140
Using columns: title_masked + full_text_masked

Processing: tvnet_reliable_dedup.csv
Saved: tvnet_reliable_dedup_standardized.csv
Rows: 156
Using columns: title_clean + full_text_clean

All files proc

# 2. MERGE clean files into masted dataset

*   loads standartized files
*   validates columns
*   combines frames
*   generates corpus summary





In [ ]:
%%writefile step2_master_merge.py
from pathlib import Path
import pandas as pd

INPUT_DIR = Path("/content/standardized")
OUTPUT_FILE = Path("/content/master_corpus.csv")

EXPECTED_COLUMNS = [
    "id",
    "title",
    "full_text",
    "source",
    "publication_date",
    "topic",
    "class_label",
    "url",
    "text_length",
]

def load_all_files():
    files = sorted(INPUT_DIR.glob("*_standardized.csv"))

    if not files:
        print("No standardized files found.")
        return None

    print(f"Found {len(files)} standardized files\n")

    dfs = []

    for file in files:
        print(f"Loading: {file.name}")
        df = pd.read_csv(file)

        if list(df.columns) != EXPECTED_COLUMNS:
            raise ValueError(
                f"Column mismatch in {file.name}\n"
                f"Expected: {EXPECTED_COLUMNS}\n"
                f"Found: {list(df.columns)}"
            )

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def corpus_report(df):

    print("\n=== CORPUS SUMMARY ===")

    print(f"Total rows: {len(df)}")

    print("\nClass distribution:")
    print(df["class_label"].value_counts())

    print("\nSources:")
    print(df["source"].value_counts())

    empty_titles = (df["title"].fillna("").str.strip() == "").sum()
    empty_texts = (df["full_text"].fillna("").str.strip() == "").sum()

    print("\nEmpty fields:")
    print(f"Empty titles: {empty_titles}")
    print(f"Empty full_text: {empty_texts}")


def main():

    df = load_all_files()

    if df is None:
        return

    corpus_report(df)

    df.to_csv(OUTPUT_FILE, index=False)

    print(f"\nSaved master corpus: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Writing step2_master_merge.py


In [ ]:
!python step2_master_merge.py

Found 7 standardized files

Loading: delfi_atmaskots_dedup_standardized.csv
Loading: delfi_reliable_dedup_standardized.csv
Loading: lsm_arzemes_dedup_standardized.csv
Loading: ltv_reliable_dedup_standardized.csv
Loading: rebaltica_recheck_dedup_standardized.csv
Loading: tvnet_faktomats_dedup_standardized.csv
Loading: tvnet_reliable_dedup_standardized.csv

=== CORPUS SUMMARY ===
Total rows: 2122

Class distribution:
class_label
0    1111
1    1011
Name: count, dtype: int64

Sources:
source
delfi (bizness)          577
re:baltica (re:check)    441
delfi_atmaskots          430
lsm                      348
tvnet                    156
tvnet (faktomats)        140
ltv                       30
Name: count, dtype: int64

Empty fields:
Empty titles: 0
Empty full_text: 0

Saved master corpus: /content/master_corpus.csv


# 3. DEDUBLICATION

In [ ]:
%%writefile step3_corpus_dedup.py
import pandas as pd

INPUT_FILE = "/content/master_corpus.csv"
OUTPUT_FILE = "/content/master_corpus_dedup.csv"


df = pd.read_csv(INPUT_FILE)

rows_before = len(df)

print("Rows before dedup:", rows_before)

# --------------------------------
# 0 Remove bad titles
# --------------------------------

bad_titles = [
    "krievijas iebrukums ukrainā"
]

df["title_clean"] = df["title"].str.lower().str.strip()
df = df[~df["title_clean"].isin(bad_titles)]

print("After removing bad titles:", len(df))


# --------------------------------
# 1 Deduplicate by URL
# --------------------------------

df = df.drop_duplicates(subset=["url"])

rows_after_url = len(df)

print("After URL dedup:", rows_after_url)


# --------------------------------
# 2 Deduplicate by full_text
# --------------------------------

df = df.drop_duplicates(subset=["full_text"])

rows_after_text = len(df)

print("After text dedup:", rows_after_text)


# --------------------------------
# Save
# --------------------------------

df.to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)

print("\nDedup summary")
print("Removed rows:", rows_before - rows_after_text)
print("Final rows:", rows_after_text)

Overwriting step3_corpus_dedup.py


In [ ]:
!python step3_corpus_dedup.py

Rows before dedup: 2122
After removing bad titles: 2050
After URL dedup: 2050
After text dedup: 2050

Saved: /content/master_corpus_dedup.csv

Dedup summary
Removed rows: 72
Final rows: 2050


# 4. REMOVE NEAR DUBLICATES

In [ ]:
%%writefile step4_similarity_dedup.py
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

INPUT_FILE = "/content/master_corpus_dedup.csv"
OUTPUT_FILE = "/content/master_corpus_final.csv"

SIM_THRESHOLD = 0.95


df = pd.read_csv(INPUT_FILE)

print("Rows before similarity dedup:", len(df))

texts = df["full_text"].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(
    stop_words=None,
    max_features=20000,
)

tfidf = vectorizer.fit_transform(texts)
sim_matrix = cosine_similarity(tfidf)

duplicates = set()
duplicate_pairs = []

for i in range(len(df)):
    if i in duplicates:
        continue
    for j in range(i + 1, len(df)):
        if sim_matrix[i, j] > SIM_THRESHOLD:
            duplicates.add(j)
            duplicate_pairs.append({
                "kept_index": i,
                "removed_index": j,
                "similarity": sim_matrix[i, j],
                "kept_class": df.iloc[i].get("class_label"),
                "removed_class": df.iloc[j].get("class_label"),
                "kept_title": df.iloc[i].get("title"),
                "removed_title": df.iloc[j].get("title"),
            })

if duplicate_pairs:
    print("\n=== REMOVED DUPLICATES WITH MATCHES ===")
    for pair in duplicate_pairs:
        print("\n-----------------------------------")
        print(f"REMOVED INDEX: {pair['removed_index']}")
        print(f"REMOVED CLASS: {pair['removed_class']}")
        print(f"REMOVED TITLE: {pair['removed_title']}")
        print(f"KEPT INDEX: {pair['kept_index']}")
        print(f"KEPT CLASS: {pair['kept_class']}")
        print(f"KEPT TITLE: {pair['kept_title']}")
        print(f"SIMILARITY: {pair['similarity']:.4f}")

df_clean = df.drop(index=list(duplicates))

print("\nRemoved near duplicates:", len(duplicates))
print("Final rows:", len(df_clean))

df_clean.to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)

Overwriting step4_similarity_dedup.py


In [ ]:
!python step4_similarity_dedup.py

Rows before similarity dedup: 2050

=== REMOVED DUPLICATES WITH MATCHES ===

-----------------------------------
REMOVED INDEX: 2008
REMOVED CLASS: 0
REMOVED TITLE: Eiropas Rekonstrukcijas un attīstības banka palielinājusi visu Baltijas valstu ekonomikas izaugsmes prognozes
KEPT INDEX: 525
KEPT CLASS: 0
KEPT TITLE: ERAB: Latvijas ekonomika aug straujāk, nekā gaidīts
SIMILARITY: 1.0000

-----------------------------------
REMOVED INDEX: 1130
REMOVED CLASS: 0
REMOVED TITLE: Irāna raida jaunu atbildes triecienu vilni pret Izraēlu. 5. marts (Teksta tiešraides arhīvs)
KEPT INDEX: 1121
KEPT CLASS: 0
KEPT TITLE: Tramps pieprasa Irānas bezierunu kapitulāciju. 6. marts (Teksta tiešraides arhīvs)
SIMILARITY: 1.0000

-----------------------------------
REMOVED INDEX: 1882
REMOVED CLASS: 1
REMOVED TITLE: Vai ūdenim ir atmiņa? Vai no mums slēpj gaidāmos karus par ūdeni? Pārbaudām Brieža izteikumus
KEPT INDEX: 1873
KEPT CLASS: 1
KEPT TITLE: Vai taisnība, ka Latvijā pabalsts par vienu bērnu ir tikai 

# CHECK CLASS DISTRIBUTION

In [ ]:
import pandas as pd

df = pd.read_csv("/content/master_corpus_final.csv")

print("Total rows:", len(df))


# --------------------------------
# Class distribution
# --------------------------------
print("\n=== CLASS DISTRIBUTION ===")
print(df["class_label"].value_counts())


# --------------------------------
# Source x Class
# --------------------------------
print("\n=== SOURCE x CLASS ===")
print(pd.crosstab(df["source"], df["class_label"]))


# --------------------------------
# Source x Class (%)
# --------------------------------
print("\n=== SOURCE x CLASS (%) ===")
print(pd.crosstab(df["source"], df["class_label"], normalize="index").round(3))

Total rows: 2042

=== CLASS DISTRIBUTION ===
class_label
0    1037
1    1005
Name: count, dtype: int64

=== SOURCE x CLASS ===
class_label              0    1
source                         
delfi (bizness)        577    0
delfi_atmaskots          0  430
lsm                    275    0
ltv                     30    0
re:baltica (re:check)    0  441
tvnet                  155    0
tvnet (faktomats)        0  134

=== SOURCE x CLASS (%) ===
class_label              0    1
source                         
delfi (bizness)        1.0  0.0
delfi_atmaskots        0.0  1.0
lsm                    1.0  0.0
ltv                    1.0  0.0
re:baltica (re:check)  0.0  1.0
tvnet                  1.0  0.0
tvnet (faktomats)      0.0  1.0


# 5. CREATE MODEL INPUT FILE

In [ ]:
%%writefile step5_model_ready_dataset.py
import pandas as pd

INPUT_FILE = "/content/master_corpus_final.csv"
OUTPUT_FILE = "/content/model_ready_dataset.csv"

KEEP_COLUMNS = [
    "title",
    "full_text",
    "class_label",
]

df = pd.read_csv(INPUT_FILE)

print("Rows before:", len(df))
print("Columns before:", list(df.columns))

df = df[KEEP_COLUMNS].copy()

# Basic safety cleanup
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["full_text"] = df["full_text"].fillna("").astype(str).str.strip()
df["class_label"] = pd.to_numeric(df["class_label"], errors="coerce").astype(int)

# Remove fully empty texts just in case
df = df[df["full_text"] != ""].copy()

print("\nRows after:", len(df))
print("Columns after:", list(df.columns))

df.to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)

Writing step5_model_ready_dataset.py


In [ ]:
!python step5_model_ready_dataset.py

Rows before: 2042
Columns before: ['id', 'title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'text_length', 'title_clean']

Rows after: 2042
Columns after: ['title', 'full_text', 'class_label']

Saved: /content/model_ready_dataset.csv


# 6. SPLIT TEST/TRAIN/VAL

In [ ]:
%%writefile step6_split_dataset.py
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_FILE = "/content/model_ready_dataset.csv"

TRAIN_FILE = "/content/train.csv"
VAL_FILE = "/content/validation.csv"
TEST_FILE = "/content/test.csv"

RANDOM_STATE = 42

df = pd.read_csv(INPUT_FILE)

print("Total rows:", len(df))
print("\nFull dataset class distribution:")
print(df["class_label"].value_counts(normalize=False))
print(df["class_label"].value_counts(normalize=True))

# 70% train, 30% temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["class_label"],
    random_state=RANDOM_STATE,
)

# 15% val, 15% test from total -> split temp 50/50
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["class_label"],
    random_state=RANDOM_STATE,
)

print("\nRows by split:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

print("\nTrain class distribution:")
print(train_df["class_label"].value_counts(normalize=False))
print(train_df["class_label"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(val_df["class_label"].value_counts(normalize=False))
print(val_df["class_label"].value_counts(normalize=True))

print("\nTest class distribution:")
print(test_df["class_label"].value_counts(normalize=False))
print(test_df["class_label"].value_counts(normalize=True))

train_df.to_csv(TRAIN_FILE, index=False)
val_df.to_csv(VAL_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

print("\nSaved files:")
print(TRAIN_FILE)
print(VAL_FILE)
print(TEST_FILE)

Overwriting step6_split_dataset.py


In [ ]:
!python step6_split_dataset.py

Total rows: 2042

Full dataset class distribution:
class_label
0    1037
1    1005
Name: count, dtype: int64
class_label
0    0.507835
1    0.492165
Name: proportion, dtype: float64

Rows by split:
Train: 1429
Validation: 306
Test: 307

Train class distribution:
class_label
0    726
1    703
Name: count, dtype: int64
class_label
0    0.508048
1    0.491952
Name: proportion, dtype: float64

Validation class distribution:
class_label
0    155
1    151
Name: count, dtype: int64
class_label
0    0.506536
1    0.493464
Name: proportion, dtype: float64

Test class distribution:
class_label
0    156
1    151
Name: count, dtype: int64
class_label
0    0.508143
1    0.491857
Name: proportion, dtype: float64

Saved files:
/content/train.csv
/content/validation.csv
/content/test.csv
